In [ ]:
from pathlib import Path

import pandas as pd
from matplotlib import pyplot as plt

from posterior_predictive_across_condition_summary import (
    build_posterior_predictive_condition_mean_traces,
    load_posterior_predictive_summary_table,
    plot_posterior_predictive_across_conditions_summary,
)

plt.rcParams['figure.facecolor'] = 'white'


In [ ]:
CONDITION_BATCH_ROOTS = {
    # "TA_NOGO": Path("$$$_sim_batches_posterior_predictive_checks_TA_NOGO"),
    # "HAND_NOGO": Path("$$$_sim_batches_posterior_predictive_checks_HAND_NOGO"),
    # "HAND_GO": Path("$$$_sim_batches_posterior_predictive_checks_HAND_GO"),
    "TA_NOGO": Path("$$$_sim_batches_posterior_predictive_checks_TA_NOGO_sbi_with_all_features"),
    "HAND_NOGO": Path("$$$_sim_batches_posterior_predictive_checks_HAND_NOGO_sbi_with_all_features"),
    "HAND_GO": Path("$$$_sim_batches_posterior_predictive_checks_HAND_GO_sbi_with_all_features"),
}

SUMMARY_SUBDIR_NAME = "parameter_set_summaries"
CONDITION_ORDER = ["HAND_GO","HAND_NOGO","TA_NOGO"]

DISPLAY_XLIM_REL_CUE_S = (-4.0, 1.0)
TRACE_REFERENCE_WINDOW_REL_CUE_S = (-4.0, -2.0)

SELECTED_PARAMETER_SET_IDS = None
PARAMETER_SET_ID_RANGE = None

OUTPUT_PATH = Path("$$$_posterior_predictive_checks_results_GO-NOGO_sbi_with_all_features") / SUMMARY_SUBDIR_NAME / "posterior_predictive_across_conditions_summary.png"

EXPORT_SUMMARY_TABLE = True
SUMMARY_TABLE_OUTPUT_DIR = Path("$$$_posterior_predictive_checks_results_GO-NOGO_sbi_with_all_features") / SUMMARY_SUBDIR_NAME
SUMMARY_TABLE_CSV_NAME = "posterior_predictive_across_conditions_summary.csv"
SUMMARY_TABLE_PKL_NAME = "posterior_predictive_across_conditions_summary.pkl"

RUN_FULL_TRACE_SUMMARY_FIGURE = False
REQUIRE_EXISTING_SUMMARY_TABLE_WHEN_SKIPPING = True
SUMMARY_CURVE_FEATURE_SOURCE = "load_or_compute"  # "load_or_compute", "load_only", or "compute"
FORCE_RECOMPUTE_SUMMARY_CURVES = False
SUMMARY_CURVE_TRACE_CACHE_PATH = OUTPUT_PATH.parent / "feature_match" / "posterior_predictive_condition_mean_traces.pkl"


ROW_COLORS = {
    "cst_lf": "#ff3300",
    "cst_alpha_mod": "#ffa600",
    "cst_beta_mod": "#008cff",
    "sync_trace": "#7e2179",
    "firing_rate": "#FF7300",
    "isi_cv": "#00CFAD",
}

FIGURE_STYLE = {
    "figure_size": (26, 18),
    "figure_dpi": 150,
    "curve_width_ratios": (2.0, 1.0),
    "projection_width_ratios": (1.0, 1.0),
    "condition_wspace": 0.18,
    "pair_wspace": 0.22,
    "row_hspace": 0.38,
    "participant_trace_lw": 1.0,
    "participant_trace_alpha": 0.05,
    "mean_trace_lw": 2.0,
    "mean_trace_linestyle": "-",
    "mode_trace_color": "#111111",
    "mode_trace_lw": 1.0,
    "mode_trace_alpha": 0.0,
    "mode_trace_linestyle": "-",
    "ready_lw": 1.0,
    "ready_alpha": 0.9,
    "go_lw": 1.2,
    "go_alpha": 0.9,
    "event_color": "#7A7A7A",
    "reference_line_color": "#7A7A7A",
    "reference_line_lw": 1.0,
    "reference_line_alpha": 0.9,
    "time_tick_step_s": 0.5,
    "violin_width": 0.7,
    "violin_alpha": 0.18,
    "violin_edge_lw": 1.2,
    "feature_mean_lw": 2.2,
    "feature_mean_alpha": 0.95,
    "feature_mean_half_width": 0.28,
    "feature_point_size": 50,
    "feature_point_alpha": 0.5,
    "feature_point_jitter": 0.08,
    "mode_feature_marker": "X",
    "mode_feature_marker_size": 110.0,
    "mode_feature_marker_alpha": 0.0,
    "mode_feature_marker_facecolor": "row",
    "mode_feature_marker_edgecolor": "#FFFFFF",
    "mode_feature_marker_edge_lw": 2.0,
    "zero_line_color": "#7A7A7A",
    "zero_line_lw": 1.0,
    "zero_line_alpha": 0.9,
    "condition_title_fontsize": 11,
    "condition_subtitle_fontsize": 9,
    "panel_title_fontsize": 9,
    "axis_label_fontsize": 8,
    "tick_labelsize": 9,
    "projection_marker_size": 24,
    "projection_alpha": 0.75,
    "projection_plane_alpha": 0.06,
    "projection_grid_alpha": 0.18,
    "projection_edge_color": "#555555",
    "projection_edge_lw": 1.4,
    "projection_edge_alpha": 0.9,
    "projection_elev": 22,
    "projection_azim": -54,
}

YLIM_OVERRIDES = { # ~same as in experimental figures
    "trace": {
        "cst_lf": (5, 20),
        "cst_alpha_mod": (-0.9, 3.1),
        "cst_beta_mod": (-0.9, 1.5),
        "sync_trace": (0, 0.7),
    },
    "feature": {
        "cst_lf": (-4.9, 1.9),
        "cst_alpha_mod": (-0.4, 1.1),
        "cst_beta_mod": (-0.25, 0.45),
        "sync_trace": (-0.07, 0.22),
    },
    "projection": {
        # "Firing-rate moments": ((10, 20), (1.5, 4), (-1.5, 1)),
        # "ISI-CV moments": ((0.05, 0.275), (0.025, 0.175), (1, 4)),
    },
}


In [ ]:
summary_table_path = Path(SUMMARY_TABLE_OUTPUT_DIR) / SUMMARY_TABLE_PKL_NAME

if RUN_FULL_TRACE_SUMMARY_FIGURE:
    fig, summary_info, across_condition_df = plot_posterior_predictive_across_conditions_summary(
        condition_batch_roots=CONDITION_BATCH_ROOTS,
        summary_subdir_name=SUMMARY_SUBDIR_NAME,
        output_path=OUTPUT_PATH,
        condition_order=CONDITION_ORDER,
        display_window_rel_cue_s=DISPLAY_XLIM_REL_CUE_S,
        selected_parameter_set_ids=SELECTED_PARAMETER_SET_IDS,
        parameter_set_id_range=PARAMETER_SET_ID_RANGE,
        row_colors=ROW_COLORS,
        style=FIGURE_STYLE,
        ylim_overrides=YLIM_OVERRIDES,
        trace_reference_window_rel_cue_s=TRACE_REFERENCE_WINDOW_REL_CUE_S,
        export_summary_table=EXPORT_SUMMARY_TABLE,
        summary_table_output_dir=SUMMARY_TABLE_OUTPUT_DIR,
        summary_table_csv_name=SUMMARY_TABLE_CSV_NAME,
        summary_table_pickle_name=SUMMARY_TABLE_PKL_NAME,
    )
else:
    if not summary_table_path.exists() and REQUIRE_EXISTING_SUMMARY_TABLE_WHEN_SKIPPING:
        raise FileNotFoundError(
            f"RUN_FULL_TRACE_SUMMARY_FIGURE=False, but no saved summary table exists at {summary_table_path}. "
            "Set RUN_FULL_TRACE_SUMMARY_FIGURE=True once to generate it, or disable REQUIRE_EXISTING_SUMMARY_TABLE_WHEN_SKIPPING."
        )
    if summary_table_path.exists():
        across_condition_df = load_posterior_predictive_summary_table(summary_table_path)
        fig = None
        summary_info = {
            "loaded_summary_table": str(summary_table_path),
            "figure_path": str(OUTPUT_PATH) if Path(OUTPUT_PATH).exists() else None,
            "n_condition_records": int(len(across_condition_df)),
            "run_full_trace_summary_figure": False,
        }
    else:
        fig, summary_info, across_condition_df = plot_posterior_predictive_across_conditions_summary(
            condition_batch_roots=CONDITION_BATCH_ROOTS,
            summary_subdir_name=SUMMARY_SUBDIR_NAME,
            output_path=OUTPUT_PATH,
            condition_order=CONDITION_ORDER,
            display_window_rel_cue_s=DISPLAY_XLIM_REL_CUE_S,
            selected_parameter_set_ids=SELECTED_PARAMETER_SET_IDS,
            parameter_set_id_range=PARAMETER_SET_ID_RANGE,
            row_colors=ROW_COLORS,
            style=FIGURE_STYLE,
            ylim_overrides=YLIM_OVERRIDES,
            trace_reference_window_rel_cue_s=TRACE_REFERENCE_WINDOW_REL_CUE_S,
            export_summary_table=EXPORT_SUMMARY_TABLE,
            summary_table_output_dir=SUMMARY_TABLE_OUTPUT_DIR,
            summary_table_csv_name=SUMMARY_TABLE_CSV_NAME,
            summary_table_pickle_name=SUMMARY_TABLE_PKL_NAME,
        )

summary_info
across_condition_df.head()


In [ ]:
# Posterior-predictive feature match tables and summary figures
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

from experimental_across_condition_summary import ROW_SPECS
from parameter_set_batch_summary import fit_baseline_spline_and_extract_feature
from shared_run_analysis import (
    DEFAULT_OBSERVATION_WINDOW_DEFS_REL_CUE_S,
    OBS_WINDOW_ALIAS_TO_CODE,
    compute_window_stat_features,
)

SELECTED_FEATURES = [
    # Burst Features
    "OBS_BURST_FEATURES_resid_cst_low_frequency",
    "OBS_BURST_FEATURES_resid_cst_alpha_modulation_hilbert_zscore",
    "OBS_BURST_FEATURES_resid_cst_beta_modulation_hilbert_zscore",
    "OBS_BURST_FEATURES_resid_sync_coincidence_curve",

    # Trend Features
    "OBS_TREND_FEATURE_cst_low_frequency",
    "OBS_TREND_FEATURE_cst_alpha_modulation_hilbert_zscore",
    "OBS_TREND_FEATURE_cst_beta_modulation_hilbert_zscore",
    "OBS_TREND_FEATURE_sync_coincidence_curve",

    # Baseline window
    "OBS_WINDOW_m3_to_m2_CST_LF_raw_mean",
    "OBS_WINDOW_m3_to_m2_ALPHA_mod_z_mean",
    "OBS_WINDOW_m3_to_m2_BETA_mod_z_mean",
    "OBS_WINDOW_m3_to_m2_SYNC_mean",
    "OBS_WINDOW_m3_to_m2_CST_LF_raw_min",
    "OBS_WINDOW_m3_to_m2_ALPHA_mod_z_min",
    "OBS_WINDOW_m3_to_m2_BETA_mod_z_min",
    "OBS_WINDOW_m3_to_m2_SYNC_min",
    "OBS_WINDOW_m3_to_m2_CST_LF_raw_max",
    "OBS_WINDOW_m3_to_m2_ALPHA_mod_z_max",
    "OBS_WINDOW_m3_to_m2_BETA_mod_z_max",
    "OBS_WINDOW_m3_to_m2_SYNC_max",

    # Post-ready cue window
    "OBS_WINDOW_m1_to_0_CST_LF_raw_mean",
    "OBS_WINDOW_m1_to_0_ALPHA_mod_z_mean",
    "OBS_WINDOW_m1_to_0_BETA_mod_z_mean",
    "OBS_WINDOW_m1_to_0_SYNC_mean",
    "OBS_WINDOW_m1_to_0_CST_LF_raw_min",
    "OBS_WINDOW_m1_to_0_ALPHA_mod_z_min",
    "OBS_WINDOW_m1_to_0_BETA_mod_z_min",
    "OBS_WINDOW_m1_to_0_SYNC_min",
    "OBS_WINDOW_m1_to_0_CST_LF_raw_max",
    "OBS_WINDOW_m1_to_0_ALPHA_mod_z_max",
    "OBS_WINDOW_m1_to_0_BETA_mod_z_max",
    "OBS_WINDOW_m1_to_0_SYNC_max",

    # Post-go/no-go cue window
    "OBS_WINDOW_0_to_1_CST_LF_raw_mean",
    "OBS_WINDOW_0_to_1_ALPHA_mod_z_mean",
    "OBS_WINDOW_0_to_1_BETA_mod_z_mean",
    "OBS_WINDOW_0_to_1_SYNC_mean",
    "OBS_WINDOW_0_to_1_CST_LF_raw_min",
    "OBS_WINDOW_0_to_1_ALPHA_mod_z_min",
    "OBS_WINDOW_0_to_1_BETA_mod_z_min",
    "OBS_WINDOW_0_to_1_SYNC_min",
    "OBS_WINDOW_0_to_1_CST_LF_raw_max",
    "OBS_WINDOW_0_to_1_ALPHA_mod_z_max",
    "OBS_WINDOW_0_to_1_BETA_mod_z_max",
    "OBS_WINDOW_0_to_1_SYNC_max",
    "OBS_WINDOW_0_to_1_CST_LF_raw_t_max_rel_cue_s",
    "OBS_WINDOW_0_to_1_CST_LF_raw_t_min_rel_cue_s",
    "OBS_WINDOW_0_to_1_ALPHA_mod_z_t_max_rel_cue_s",
    "OBS_WINDOW_0_to_1_BETA_mod_z_t_max_rel_cue_s",
    "OBS_WINDOW_0_to_1_SYNC_t_max_rel_cue_s",
]

EXPERIMENTAL_FEATURE_TABLE_PATH = Path("EXP_DATA_ANALYZED_HDF5/_batch_summaries/experimental_batch_feature_summary.pkl")
FEATURE_MATCH_OUTPUT_DIR = OUTPUT_PATH.parent / "feature_match"
FEATURE_MATCH_STANDARDIZE = True  # Legacy alias: True means FEATURE_MATCH_METRIC_VALUE_SPACE="z" unless overridden below.
FEATURE_MATCH_METRIC_VALUE_SPACE = "experimental_abs_zero"  # "raw", "z", or "experimental_abs_zero"
FEATURE_MATCH_SKIP_MISSING_FEATURES = True

FEATURE_VALUE_SCATTER_VALUE_SPACE = "raw" # "raw"  # "raw", "z", or "experimental_abs_zero"
FEATURE_VALUE_SCATTER_N_COLS = 4
FEATURE_VALUE_SCATTER_FIGSIZE_PER_PANEL = (4.2, 3.0)
FEATURE_VALUE_SCATTER_POINT_SIZE = 34
FEATURE_VALUE_SCATTER_POINT_ALPHA = 0.28
FEATURE_VALUE_SCATTER_POINT_JITTER = 0.09
FEATURE_VALUE_SCATTER_EXPERIMENTAL_MARKER = "X"
FEATURE_VALUE_SCATTER_EXPERIMENTAL_SIZE = 115
FEATURE_VALUE_SCATTER_EXPERIMENTAL_EDGE_COLOR = "black"
FEATURE_VALUE_SCATTER_EXPERIMENTAL_EDGE_LW = 1.1

FEATURES_COLORS = {
    "cst_low_frequency": "#ff3300",
    "cst_alpha": "#ffa600",
    "cst_beta": "#008cff",
    "sync_coincidence": "#852680",
    "LF": "#ff3300",
    "ALPHA": "#ffa600",
    "BETA": "#008cff",
    "SYNC": "#852680",
}
EXPERIMENTAL_COLORS = {
    "HAND_GO": "#ffbb00",
    "HAND_NOGO": "#ff6600",
    "TA_NOGO": "#ff0080",
}
FEATURE_BY_CONDITION_VALUE_SPACE = "experimental_abs_zero"  # "raw", "z", or "experimental_abs_zero"
FEATURE_BY_CONDITION_FIG_WIDTH_PER_FEATURE = 1.5
FEATURE_BY_CONDITION_MIN_FIG_WIDTH = 16.0
FEATURE_BY_CONDITION_ROW_HEIGHT = 4.3
FEATURE_BY_CONDITION_POINT_SIZE = 28
FEATURE_BY_CONDITION_POINT_ALPHA = 0.30
FEATURE_BY_CONDITION_POINT_JITTER = 0.11
FEATURE_BY_CONDITION_MEAN_BAR_HALF_WIDTH = 0.24
FEATURE_BY_CONDITION_MEAN_BAR_LW = 2.4
FEATURE_BY_CONDITION_MEAN_TRACE_BAR_LW = 2.0
FEATURE_BY_CONDITION_SHOW_MEAN_TRACE_BAR = False
FEATURE_BY_CONDITION_YLIM = (-2.0, 3.5) # (-2.0, 2.0)
FEATURE_BY_CONDITION_XTICK_LABEL_MODE = "full"  # "full" or "short"
FEATURE_BY_CONDITION_METRIC_TEXT_X = 0.03
FEATURE_BY_CONDITION_METRIC_TEXT_Y = 0.04
FEATURE_BY_CONDITION_METRIC_TEXT_FONTSIZE = 9
FEATURE_BY_CONDITION_FEATURE_ERROR_LABEL_Y = 1.12
FEATURE_BY_CONDITION_FEATURE_ERROR_LABEL_FONTSIZE = 6
FEATURE_BY_CONDITION_SHOW_METRIC_VIOLIN_PANEL = True
FEATURE_BY_CONDITION_FEATURE_PANEL_WIDTH_RATIO = 5.2
FEATURE_BY_CONDITION_METRIC_PANEL_WIDTH_RATIO = 1.0
FEATURE_BY_CONDITION_METRIC_VIOLIN_YLIM = (0.0, 1.0)
FEATURE_BY_CONDITION_METRIC_COLORS = {
    "r": "#83e266",
    "r2_corr": "#c68dff",
}
FEATURE_BY_CONDITION_METRIC_POINT_ALPHA = 0.35
FEATURE_BY_CONDITION_METRIC_POINT_SIZE = 24
FEATURE_BY_CONDITION_METRIC_MEAN_LW = 2.2
FEATURE_BY_CONDITION_METRIC_SUMMARY_MARKER = "X"
FEATURE_BY_CONDITION_METRIC_SUMMARY_SIZE = 95
FEATURE_BY_CONDITION_EXPERIMENTAL_TARGET_HALF_WIDTH = 0.30
FEATURE_BY_CONDITION_EXPERIMENTAL_TARGET_LW = 2.2
FEATURE_BY_CONDITION_EXPERIMENTAL_TARGET_COLOR = "black"

FEATURE_MATCH_CONDITION_COLORS = {
    **EXPERIMENTAL_COLORS,
    "Overall": "#777777",
}


def _load_sim_summary_table():
    if "across_condition_df" in globals() and isinstance(across_condition_df, pd.DataFrame) and not across_condition_df.empty:
        return across_condition_df.copy()
    fallback_path = Path(SUMMARY_TABLE_OUTPUT_DIR) / SUMMARY_TABLE_PKL_NAME
    if not fallback_path.exists():
        raise FileNotFoundError(
            "No in-memory across_condition_df is available and the exported posterior-predictive "
            f"summary table does not exist: {fallback_path}"
        )
    return pd.read_pickle(fallback_path)


def _safe_pearson(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if np.count_nonzero(mask) < 2:
        return np.nan, int(np.count_nonzero(mask))
    x = x[mask]
    y = y[mask]
    if np.nanstd(x) <= 1e-12 or np.nanstd(y) <= 1e-12:
        return np.nan, int(mask.sum())
    return float(np.corrcoef(x, y)[0, 1]), int(mask.sum())


def _vector_metrics(sim_values, exp_values):
    sim_values = np.asarray(sim_values, dtype=float)
    exp_values = np.asarray(exp_values, dtype=float)
    mask = np.isfinite(sim_values) & np.isfinite(exp_values)
    n = int(np.count_nonzero(mask))
    if n == 0:
        return {"r": np.nan, "r2_corr": np.nan, "rmse_z": np.nan, "mae_z": np.nan, "n": 0}
    diff = sim_values[mask] - exp_values[mask]
    r, _ = _safe_pearson(sim_values, exp_values)
    return {
        "r": r,
        "r2_corr": np.nan if not np.isfinite(r) else float(r * r),
        "rmse_z": float(np.sqrt(np.nanmean(diff * diff))),
        "mae_z": float(np.nanmean(np.abs(diff))),
        "n": n,
    }


def _feature_group(feature_name):
    name = str(feature_name)
    if "BURST_FEATURES" in name:
        return "Burst"
    if "TREND_FEATURE" in name:
        return "Trend"
    if "FIRING_STATISTICS" in name:
        return "Firing"
    if "BASELINE" in name:
        return "Baseline"
    if "SPECTRUM" in name:
        return "Spectrum"
    if "OBS_WINDOW_m3_to_m2" in name:
        return "Window -3:-2"
    if "OBS_WINDOW_m1_to_0" in name:
        return "Window -1:0"
    if "OBS_WINDOW_0_to_1" in name:
        return "Window 0:1"
    if "OBS_WINDOW" in name:
        return "Window"
    return "Other"


def _short_feature_label(feature_name):
    label = str(feature_name)
    for prefix in ("OBS_BURST_FEATURES_resid_", "OBS_TREND_FEATURE_", "OBS_WINDOW_", "OBS_BASELINE_", "OBS_FIRING_STATISTICS_"):
        label = label.replace(prefix, "")
    return label.replace("cst_", "").replace("modulation_hilbert_zscore", "mod z")


def _feature_color(feature_name):
    feature_text = str(feature_name)
    for key, color in FEATURES_COLORS.items():
        if key in feature_text:
            return color
    return "#555555"


def _feature_tick_label(feature_name):
    if str(FEATURE_BY_CONDITION_XTICK_LABEL_MODE).lower() == "full":
        return str(feature_name)
    return _short_feature_label(feature_name).replace("raw", "").replace("_", " ")


def _parameter_set_key(row):
    is_mode = bool(row.get("is_posterior_mode", False))
    if is_mode:
        return "posterior_mode"
    value = row.get("parameter_set_id", np.nan)
    if pd.isna(value):
        return "parameter_set_unknown"
    return f"parameter_set_{int(value):04d}"


def _export_table(df, stem):
    csv_path = FEATURE_MATCH_OUTPUT_DIR / f"{stem}.csv"
    pkl_path = FEATURE_MATCH_OUTPUT_DIR / f"{stem}.pkl"
    df.to_csv(csv_path, index=False)
    df.to_pickle(pkl_path)
    return {"csv": str(csv_path), "pkl": str(pkl_path)}


FEATURE_MATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sim_summary_df = _load_sim_summary_table()
exp_participant_df = pd.read_pickle(EXPERIMENTAL_FEATURE_TABLE_PATH)

sim_missing = [f for f in SELECTED_FEATURES if f not in sim_summary_df.columns]
exp_missing = [f for f in SELECTED_FEATURES if f not in exp_participant_df.columns]
if sim_missing or exp_missing:
    msg = f"Missing selected features | simulation={sim_missing} | experimental={exp_missing}"
    if FEATURE_MATCH_SKIP_MISSING_FEATURES:
        warnings.warn(msg + "; skipping missing features.")
    else:
        raise KeyError(msg)
shared_features = [f for f in SELECTED_FEATURES if f in sim_summary_df.columns and f in exp_participant_df.columns]
if not shared_features:
    raise ValueError("No selected features are shared by simulation and experimental summary tables.")

sim_feature_table = pd.DataFrame({
    "condition_id": sim_summary_df["SIM_SUMMARY_condition_id"].astype(str),
    "parameter_set_id": pd.to_numeric(sim_summary_df.get("SIM_SUMMARY_parameter_set_id"), errors="coerce"),
    "is_posterior_mode": sim_summary_df.get("SIM_SUMMARY_is_posterior_mode", False).astype(bool),
    "mode_label": sim_summary_df.get("SIM_SUMMARY_mode_observation_label", pd.Series(np.nan, index=sim_summary_df.index)),
    "parameter_set_dir": sim_summary_df.get("SIM_SUMMARY_parameter_set_dir", pd.Series("", index=sim_summary_df.index)),
    "n_repeats": pd.to_numeric(sim_summary_df.get("SIM_SUMMARY_n_repeats", pd.Series(np.nan, index=sim_summary_df.index)), errors="coerce"),
})
for feature in shared_features:
    sim_feature_table[feature] = pd.to_numeric(sim_summary_df[feature], errors="coerce")
sim_feature_table["parameter_set_key"] = sim_feature_table.apply(_parameter_set_key, axis=1)

exp_meta_cols = [
    c for c in ["DATASET_CONTEXT_participant_id", "DATASET_CONTEXT_session_tag", "DATASET_CONTEXT_condition_id"]
    if c in exp_participant_df.columns
]
exp_participant_condition_feature_table = exp_participant_df.loc[:, exp_meta_cols + shared_features].copy()
for feature in shared_features:
    exp_participant_condition_feature_table[feature] = pd.to_numeric(exp_participant_condition_feature_table[feature], errors="coerce")
exp_participant_condition_feature_table = exp_participant_condition_feature_table.rename(
    columns={"DATASET_CONTEXT_condition_id": "condition_id"}
)
exp_participant_condition_feature_table["condition_id"] = exp_participant_condition_feature_table["condition_id"].astype(str)

exp_condition_feature_table = (
    exp_participant_condition_feature_table
    .groupby("condition_id", as_index=False)[shared_features]
    .mean(numeric_only=True)
)

expected_conditions = [str(c) for c in CONDITION_ORDER]
missing_sim_conditions = sorted(set(expected_conditions) - set(sim_feature_table["condition_id"].dropna().astype(str)))
missing_exp_conditions = sorted(set(expected_conditions) - set(exp_condition_feature_table["condition_id"].dropna().astype(str)))
if missing_sim_conditions:
    warnings.warn(f"No posterior-predictive simulation rows found for conditions: {missing_sim_conditions}")
if missing_exp_conditions:
    warnings.warn(f"No experimental condition means found for conditions: {missing_exp_conditions}")

scale_means = exp_participant_condition_feature_table[shared_features].mean(axis=0, skipna=True)
scale_stds = exp_participant_condition_feature_table[shared_features].std(axis=0, skipna=True, ddof=0)
abs_zero_scales = exp_participant_condition_feature_table[shared_features].abs().mean(axis=0, skipna=True)
valid_scale_features = [
    f for f in shared_features
    if np.isfinite(scale_means[f]) and np.isfinite(scale_stds[f]) and float(scale_stds[f]) > 0.0
]
valid_abs_zero_features = [
    f for f in shared_features
    if np.isfinite(abs_zero_scales[f]) and float(abs_zero_scales[f]) > 0.0
]
invalid_scale_features = [f for f in shared_features if f not in valid_scale_features]
invalid_abs_zero_features = [f for f in shared_features if f not in valid_abs_zero_features]
if invalid_scale_features:
    warnings.warn(f"Skipping features with zero/non-finite experimental SD for z-scored metrics: {invalid_scale_features}")
if invalid_abs_zero_features:
    warnings.warn(f"Skipping features with zero/non-finite experimental mean absolute distance from zero: {invalid_abs_zero_features}")
metric_value_space = str(FEATURE_MATCH_METRIC_VALUE_SPACE).lower()
if metric_value_space not in {"raw", "z", "experimental_abs_zero"}:
    raise ValueError("FEATURE_MATCH_METRIC_VALUE_SPACE must be 'raw', 'z', or 'experimental_abs_zero'.")
if metric_value_space == "z" and not valid_scale_features:
    raise ValueError("No selected feature has a finite nonzero experimental participant-condition SD.")
if metric_value_space == "experimental_abs_zero" and not valid_abs_zero_features:
    raise ValueError("No selected feature has a finite nonzero experimental mean absolute distance from zero.")
metric_features = {
    "raw": shared_features,
    "z": valid_scale_features,
    "experimental_abs_zero": valid_abs_zero_features,
}[metric_value_space]

sim_feature_table_z = sim_feature_table[["condition_id", "parameter_set_id", "parameter_set_key", "is_posterior_mode", "mode_label", "parameter_set_dir", "n_repeats"]].copy()
exp_condition_feature_table_z = exp_condition_feature_table[["condition_id"]].copy()
sim_feature_table_abs_zero = sim_feature_table[["condition_id", "parameter_set_id", "parameter_set_key", "is_posterior_mode", "mode_label", "parameter_set_dir", "n_repeats"]].copy()
exp_condition_feature_table_abs_zero = exp_condition_feature_table[["condition_id"]].copy()
for feature in valid_scale_features:
    sim_feature_table_z[feature] = (sim_feature_table[feature] - scale_means[feature]) / scale_stds[feature]
    exp_condition_feature_table_z[feature] = (exp_condition_feature_table[feature] - scale_means[feature]) / scale_stds[feature]
for feature in valid_abs_zero_features:
    sim_feature_table_abs_zero[feature] = sim_feature_table[feature] / abs_zero_scales[feature]
    exp_condition_feature_table_abs_zero[feature] = exp_condition_feature_table[feature] / abs_zero_scales[feature]

metric_table_by_space = {
    "raw": (sim_feature_table, exp_condition_feature_table),
    "z": (sim_feature_table_z, exp_condition_feature_table_z),
    "experimental_abs_zero": (sim_feature_table_abs_zero, exp_condition_feature_table_abs_zero),
}
metric_sim_table, metric_exp_table = metric_table_by_space[metric_value_space]
exp_by_condition = metric_exp_table.set_index("condition_id")

simulation_metric_rows = []
for _, row in metric_sim_table.iterrows():
    condition = str(row["condition_id"])
    if condition not in exp_by_condition.index:
        continue
    metrics = _vector_metrics(row[metric_features].to_numpy(dtype=float), exp_by_condition.loc[condition, metric_features].to_numpy(dtype=float))
    simulation_metric_rows.append({
        "condition_id": condition,
        "parameter_set_id": row.get("parameter_set_id", np.nan),
        "parameter_set_key": row.get("parameter_set_key", ""),
        "is_posterior_mode": bool(row.get("is_posterior_mode", False)),
        "mode_label": row.get("mode_label", np.nan),
        "r": metrics["r"],
        "r2_corr": metrics["r2_corr"],
        "rmse_z": metrics["rmse_z"],
        "mae_z": metrics["mae_z"],
        "n_features": metrics["n"],
    })
simulation_direction_metrics = pd.DataFrame(simulation_metric_rows)

feature_metric_rows = []
for (parameter_set_key, is_posterior_mode), group in metric_sim_table.groupby(["parameter_set_key", "is_posterior_mode"], dropna=False):
    group_by_condition = group.groupby("condition_id", as_index=True).mean(numeric_only=True)
    for feature in metric_features:
        common_conditions = [c for c in expected_conditions if c in group_by_condition.index and c in exp_by_condition.index]
        if not common_conditions:
            common_conditions = sorted(set(group_by_condition.index).intersection(exp_by_condition.index))
        sim_values = group_by_condition.loc[common_conditions, feature].to_numpy(dtype=float) if common_conditions else np.asarray([])
        exp_values = exp_by_condition.loc[common_conditions, feature].to_numpy(dtype=float) if common_conditions else np.asarray([])
        metrics = _vector_metrics(sim_values, exp_values)
        first = group.iloc[0]
        feature_metric_rows.append({
            "parameter_set_key": parameter_set_key,
            "parameter_set_id": first.get("parameter_set_id", np.nan),
            "is_posterior_mode": bool(is_posterior_mode),
            "mode_label": first.get("mode_label", np.nan),
            "feature": feature,
            "feature_group": _feature_group(feature),
            "feature_label": _short_feature_label(feature),
            "r": metrics["r"],
            "r2_corr": metrics["r2_corr"],
            "rmse_z": metrics["rmse_z"],
            "mae_z": metrics["mae_z"],
            "n_conditions": metrics["n"],
        })
feature_direction_metrics = pd.DataFrame(feature_metric_rows)

summary_by_condition = (
    simulation_direction_metrics
    .groupby(["condition_id", "is_posterior_mode"], dropna=False)
    .agg(
        n_rows=("parameter_set_key", "count"),
        r_mean=("r", "mean"),
        r_median=("r", "median"),
        r2_corr_mean=("r2_corr", "mean"),
        r2_corr_median=("r2_corr", "median"),
        rmse_z_mean=("rmse_z", "mean"),
        rmse_z_median=("rmse_z", "median"),
        mae_z_mean=("mae_z", "mean"),
        mae_z_median=("mae_z", "median"),
    )
    .reset_index()
)
summary_by_feature = (
    feature_direction_metrics
    .groupby(["feature_group", "feature", "is_posterior_mode"], dropna=False)
    .agg(
        n_rows=("parameter_set_key", "count"),
        r_mean=("r", "mean"),
        r_median=("r", "median"),
        r2_corr_mean=("r2_corr", "mean"),
        r2_corr_median=("r2_corr", "median"),
        rmse_z_mean=("rmse_z", "mean"),
        rmse_z_median=("rmse_z", "median"),
        mae_z_mean=("mae_z", "mean"),
        mae_z_median=("mae_z", "median"),
    )
    .reset_index()
)
summary_overall = pd.concat([
    simulation_direction_metrics.assign(metric_direction="simulation_direction")
    .groupby(["metric_direction", "is_posterior_mode"], dropna=False)
    .agg(
        n_rows=("parameter_set_key", "count"),
        r_mean=("r", "mean"),
        r_median=("r", "median"),
        r2_corr_mean=("r2_corr", "mean"),
        r2_corr_median=("r2_corr", "median"),
        rmse_z_mean=("rmse_z", "mean"),
        rmse_z_median=("rmse_z", "median"),
        mae_z_mean=("mae_z", "mean"),
        mae_z_median=("mae_z", "median"),
    )
    .reset_index(),
    feature_direction_metrics.assign(metric_direction="feature_direction")
    .groupby(["metric_direction", "is_posterior_mode"], dropna=False)
    .agg(
        n_rows=("parameter_set_key", "count"),
        r_mean=("r", "mean"),
        r_median=("r", "median"),
        r2_corr_mean=("r2_corr", "mean"),
        r2_corr_median=("r2_corr", "median"),
        rmse_z_mean=("rmse_z", "mean"),
        rmse_z_median=("rmse_z", "median"),
        mae_z_mean=("mae_z", "mean"),
        mae_z_median=("mae_z", "median"),
    )
    .reset_index(),
], ignore_index=True)

saved_feature_match_paths = {}
saved_feature_match_paths["sim_feature_raw"] = _export_table(sim_feature_table, "posterior_predictive_sim_feature_table_raw")
saved_feature_match_paths["exp_condition_raw"] = _export_table(exp_condition_feature_table, "experimental_condition_feature_table_raw")
saved_feature_match_paths["exp_participant_condition_raw"] = _export_table(exp_participant_condition_feature_table, "experimental_participant_condition_feature_table_raw")
saved_feature_match_paths["sim_feature_z"] = _export_table(sim_feature_table_z, "posterior_predictive_sim_feature_table_z")
saved_feature_match_paths["exp_condition_z"] = _export_table(exp_condition_feature_table_z, "experimental_condition_feature_table_z")
saved_feature_match_paths["sim_feature_abs_zero"] = _export_table(sim_feature_table_abs_zero, "posterior_predictive_sim_feature_table_experimental_abs_zero")
saved_feature_match_paths["exp_condition_abs_zero"] = _export_table(exp_condition_feature_table_abs_zero, "experimental_condition_feature_table_experimental_abs_zero")
saved_feature_match_paths["simulation_direction_metrics"] = _export_table(simulation_direction_metrics, "feature_match_simulation_direction_metrics")

scaler_table = pd.DataFrame({
    "feature": shared_features,
    "experimental_mean": [scale_means[f] for f in shared_features],
    "experimental_sd": [scale_stds[f] for f in shared_features],
    "experimental_mean_abs_distance_from_zero": [abs_zero_scales[f] for f in shared_features],
    "used_for_current_metric_space": [f in metric_features for f in shared_features],
    "current_metric_value_space": metric_value_space,
})
scaler_table.to_csv(FEATURE_MATCH_OUTPUT_DIR / "feature_match_experimental_scaler.csv", index=False)

# Figure summary metadata. The current notebook only generates the combined condition-by-feature figure below.
feature_match_summary_info = {
    "output_dir": str(FEATURE_MATCH_OUTPUT_DIR),
    "n_selected_features": len(SELECTED_FEATURES),
    "n_shared_features": len(shared_features),
    "n_metric_features": len(metric_features),
    "missing_simulation_features": sim_missing,
    "missing_experimental_features": exp_missing,
    "invalid_scale_features": invalid_scale_features,
    "invalid_abs_zero_features": invalid_abs_zero_features,
    "metric_value_space": metric_value_space,
    "saved_tables": saved_feature_match_paths,
}


# Summary-level posterior-predictive feature metrics and value scatter
TRACE_SIGNAL_SPECS = {
    "cst_lf": {
        "signal_family": "CST_LF_raw",
        "burst_col": "OBS_BURST_FEATURES_resid_cst_low_frequency",
        "trend_col": "OBS_TREND_FEATURE_cst_low_frequency",
    },
    "cst_alpha_mod": {
        "signal_family": "ALPHA_mod_z",
        "burst_col": "OBS_BURST_FEATURES_resid_cst_alpha_modulation_hilbert_zscore",
        "trend_col": "OBS_TREND_FEATURE_cst_alpha_modulation_hilbert_zscore",
    },
    "cst_beta_mod": {
        "signal_family": "BETA_mod_z",
        "burst_col": "OBS_BURST_FEATURES_resid_cst_beta_modulation_hilbert_zscore",
        "trend_col": "OBS_TREND_FEATURE_cst_beta_modulation_hilbert_zscore",
    },
    "sync_trace": {
        "signal_family": "SYNC",
        "burst_col": "OBS_BURST_FEATURES_resid_sync_coincidence_curve",
        "trend_col": "OBS_TREND_FEATURE_sync_coincidence_curve",
    },
}


def _first_finite_metadata_value(df, column, default=None):
    if column not in df.columns:
        return default
    values = df[column].dropna()
    if values.empty:
        return default
    return values.iloc[0]


def _coerce_tuple_metadata(value, default):
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return default
    if isinstance(value, str):
        stripped = value.strip()
        if stripped.lower() in {"", "nan", "none"}:
            return default
        try:
            import ast
            value = ast.literal_eval(stripped)
        except Exception:
            return default
    try:
        return tuple(map(float, value))
    except Exception:
        return default


def _coerce_optional_int(value, default=None):
    try:
        if value is None or not np.isfinite(float(value)):
            return default
        return int(value)
    except Exception:
        return default


def _feature_config_from_summary_table(df):
    return {
        "enabled": True,
        "fitting_window": _coerce_tuple_metadata(_first_finite_metadata_value(df, "ANALYSIS_PARAM_FEATURE_fitting_window"), (-3.0, 1.0)),
        "analysis_window": _coerce_tuple_metadata(_first_finite_metadata_value(df, "ANALYSIS_PARAM_FEATURE_analysis_window"), (0.0, 0.5)),
        "buffer_s": float(_first_finite_metadata_value(df, "ANALYSIS_PARAM_FEATURE_buffer_s", 0.05)),
        "spline_smoothing": None,
        "spline_order": _coerce_optional_int(_first_finite_metadata_value(df, "ANALYSIS_PARAM_FEATURE_spline_order"), 3),
        "n_knots": _coerce_optional_int(_first_finite_metadata_value(df, "ANALYSIS_PARAM_FEATURE_n_knots"), None),
        "trend_pre_window": _coerce_tuple_metadata(_first_finite_metadata_value(df, "ANALYSIS_PARAM_FEATURE_trend_pre_window"), (-3.0, -1.0)),
        "trend_post_window": _coerce_tuple_metadata(_first_finite_metadata_value(df, "ANALYSIS_PARAM_FEATURE_trend_post_window"), (0.0, 1.0)),
        "extrapolate": False,
    }


def _zscore_feature_table(table, features):
    out = table.copy()
    for feature in features:
        if feature in out.columns and feature in scale_means and feature in scale_stds and np.isfinite(scale_stds[feature]) and scale_stds[feature] > 0:
            out[feature] = (pd.to_numeric(out[feature], errors="coerce") - scale_means[feature]) / scale_stds[feature]
    return out


def _abs_zero_scale_feature_table(table, features):
    out = table.copy()
    for feature in features:
        if feature in out.columns and feature in abs_zero_scales and np.isfinite(abs_zero_scales[feature]) and abs_zero_scales[feature] > 0:
            out[feature] = pd.to_numeric(out[feature], errors="coerce") / abs_zero_scales[feature]
    return out


def _build_condition_mean_feature_table(source_table, features):
    sampled = source_table.loc[~source_table["is_posterior_mode"].astype(bool)].copy()
    rows = []
    for condition in CONDITION_ORDER:
        subset = sampled.loc[sampled["condition_id"].astype(str) == str(condition)]
        row = {
            "condition_id": str(condition),
            "summary_source": "mean_feature",
            "n_parameter_sets": int(len(subset)),
        }
        for feature in features:
            row[feature] = float(pd.to_numeric(subset[feature], errors="coerce").mean()) if feature in subset.columns and not subset.empty else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def _extract_mean_trace_features(trace_cache, selected_features, feature_config):
    rows = []
    generated_features = set()
    for condition in CONDITION_ORDER:
        trace_group = trace_cache.get("traces_by_condition", {}).get(str(condition), {})
        row = {
            "condition_id": str(condition),
            "summary_source": "mean_trace",
            "n_parameter_sets": np.nan,
        }
        for trace_key, spec in TRACE_SIGNAL_SPECS.items():
            trace_payload = trace_group.get(trace_key, {})
            time_rel = trace_payload.get("time")
            signal = trace_payload.get("mean")
            row["n_parameter_sets"] = max(
                0 if pd.isna(row["n_parameter_sets"]) else int(row["n_parameter_sets"]),
                int(trace_payload.get("n_parameter_sets", 0) or 0),
            )
            if time_rel is None or signal is None:
                continue
            time_rel = np.asarray(time_rel, dtype=float)
            signal = np.asarray(signal, dtype=float)
            if time_rel.shape != signal.shape or time_rel.size == 0:
                continue
            for alias, window_rel_cue_s in DEFAULT_OBSERVATION_WINDOW_DEFS_REL_CUE_S.items():
                window_code = OBS_WINDOW_ALIAS_TO_CODE[alias]
                row.update(
                    compute_window_stat_features(
                        time_rel,
                        signal,
                        window_rel_cue_s=window_rel_cue_s,
                        window_code=window_code,
                        signal_family=spec["signal_family"],
                    )
                )
            try:
                feature_result = fit_baseline_spline_and_extract_feature(
                    time_rel,
                    signal,
                    fitting_window=feature_config["fitting_window"],
                    analysis_window=feature_config["analysis_window"],
                    buffer_s=feature_config["buffer_s"],
                    spline_smoothing=feature_config["spline_smoothing"],
                    spline_order=feature_config["spline_order"],
                    n_knots=feature_config["n_knots"],
                    trend_pre_window=feature_config.get("trend_pre_window"),
                    trend_post_window=feature_config.get("trend_post_window"),
                    extrapolate=feature_config["extrapolate"],
                    return_debug_info=False,
                )
                row[spec["burst_col"]] = feature_result.feature_mean_residual
                row[spec["trend_col"]] = feature_result.trend_post_minus_pre
            except Exception as exc:
                warnings.warn(f"Could not extract mean-trace spline features for {condition} / {trace_key}: {exc}")
        generated_features.update([col for col in row if col.startswith("OBS_")])
        rows.append(row)
    table = pd.DataFrame(rows)
    for feature in selected_features:
        if feature not in table.columns:
            table[feature] = np.nan
    return table[["condition_id", "summary_source", "n_parameter_sets", *selected_features]], sorted(generated_features)


def _build_summary_level_condition_metrics(summary_tables_by_source, exp_table_by_condition, features):
    rows = []
    for source_name, table in summary_tables_by_source.items():
        for _, sim_row in table.iterrows():
            condition = str(sim_row["condition_id"])
            exp_subset = exp_table_by_condition.loc[exp_table_by_condition["condition_id"].astype(str) == condition]
            if exp_subset.empty:
                continue
            exp_row = exp_subset.iloc[0]
            available = [f for f in features if f in table.columns and f in exp_table_by_condition.columns]
            metrics = _vector_metrics(sim_row[available].to_numpy(dtype=float), exp_row[available].to_numpy(dtype=float))
            rows.append({
                "summary_source": source_name,
                "condition_id": condition,
                "r": metrics["r"],
                "r2_corr": metrics["r2_corr"],
                "rmse_z": metrics["rmse_z"],
                "mae_z": metrics["mae_z"],
                "n_features": metrics["n"],
                "n_parameter_sets": sim_row.get("n_parameter_sets", np.nan),
            })
    return pd.DataFrame(rows)


def _build_summary_level_feature_metrics(summary_tables_by_source, exp_table_by_condition, features):
    rows = []
    for source_name, table in summary_tables_by_source.items():
        for feature in features:
            sim_vals = []
            exp_vals = []
            for condition in CONDITION_ORDER:
                sim_subset = table.loc[table["condition_id"].astype(str) == str(condition)]
                exp_subset = exp_table_by_condition.loc[exp_table_by_condition["condition_id"].astype(str) == str(condition)]
                if sim_subset.empty or exp_subset.empty or feature not in table.columns or feature not in exp_table_by_condition.columns:
                    continue
                sim_vals.append(sim_subset.iloc[0].get(feature, np.nan))
                exp_vals.append(exp_subset.iloc[0].get(feature, np.nan))
            metrics = _vector_metrics(np.asarray(sim_vals, dtype=float), np.asarray(exp_vals, dtype=float))
            rows.append({
                "summary_source": source_name,
                "feature": feature,
                "feature_group": _feature_group(feature),
                "feature_label": _short_feature_label(feature),
                "r": metrics["r"],
                "r2_corr": metrics["r2_corr"],
                "rmse_z": metrics["rmse_z"],
                "mae_z": metrics["mae_z"],
                "n_conditions": metrics["n"],
            })
    return pd.DataFrame(rows)


def _plot_feature_value_scatter(sim_table, exp_table, features, *, value_space):
    n_features = len(features)
    if n_features == 0:
        return None
    n_cols = int(FEATURE_VALUE_SCATTER_N_COLS)
    n_rows = int(np.ceil(n_features / n_cols))
    fig_w = float(FEATURE_VALUE_SCATTER_FIGSIZE_PER_PANEL[0]) * n_cols
    fig_h = float(FEATURE_VALUE_SCATTER_FIGSIZE_PER_PANEL[1]) * n_rows
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_w, fig_h), squeeze=False)
    rng = np.random.default_rng(42)
    x_lookup = {condition: idx for idx, condition in enumerate(CONDITION_ORDER)}
    sampled = sim_table.loc[~sim_table["is_posterior_mode"].astype(bool)].copy()
    for ax, feature in zip(axes.ravel(), features):
        for condition in CONDITION_ORDER:
            color = FEATURE_MATCH_CONDITION_COLORS.get(condition, "#777777")
            subset = sampled.loc[sampled["condition_id"].astype(str) == str(condition)]
            y = pd.to_numeric(subset.get(feature, pd.Series(dtype=float)), errors="coerce").to_numpy(dtype=float)
            y = y[np.isfinite(y)]
            if y.size:
                x = np.full(y.shape, x_lookup[condition], dtype=float)
                x += rng.uniform(-FEATURE_VALUE_SCATTER_POINT_JITTER, FEATURE_VALUE_SCATTER_POINT_JITTER, size=y.size)
                ax.scatter(
                    x,
                    y,
                    s=FEATURE_VALUE_SCATTER_POINT_SIZE,
                    alpha=FEATURE_VALUE_SCATTER_POINT_ALPHA,
                    color=color,
                    edgecolors="none",
                    zorder=2,
                )
            exp_subset = exp_table.loc[exp_table["condition_id"].astype(str) == str(condition)]
            if not exp_subset.empty and feature in exp_subset.columns:
                exp_value = pd.to_numeric(pd.Series([exp_subset.iloc[0].get(feature)]), errors="coerce").iloc[0]
                exp_value_for_label = exp_value
                if np.isfinite(exp_value):
                    ax.scatter(
                        [x_lookup[condition]],
                        [float(exp_value)],
                        s=FEATURE_VALUE_SCATTER_EXPERIMENTAL_SIZE,
                        marker=FEATURE_VALUE_SCATTER_EXPERIMENTAL_MARKER,
                        facecolors=color,
                        edgecolors=FEATURE_VALUE_SCATTER_EXPERIMENTAL_EDGE_COLOR,
                        linewidths=FEATURE_VALUE_SCATTER_EXPERIMENTAL_EDGE_LW,
                        zorder=5,
                    )
        ax.set_title(_short_feature_label(feature), fontsize=9)
        ax.set_xticks(range(len(CONDITION_ORDER)))
        ax.set_xticklabels(CONDITION_ORDER, rotation=35, ha="right", fontsize=8)
        ax.tick_params(axis="y", labelsize=8)
        ax.grid(axis="y", alpha=0.22)
    for ax in axes.ravel()[n_features:]:
        ax.axis("off")
    fig.suptitle(f"Posterior-predictive feature values vs experimental condition means ({value_space})")
    fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.985))
    return fig


def _format_metric_value(value):
    return "nan" if not np.isfinite(value) else f"{float(value):.2f}"


def _condition_metric_text(condition, condition_metrics, simulation_metrics):
    lines = []
    for source, label in (("mean_feature", "Mean features"), ("mean_trace", "Mean curve")):
        if source == "mean_trace" and not FEATURE_BY_CONDITION_SHOW_MEAN_TRACE_BAR:
            continue
        subset = condition_metrics.loc[
            (condition_metrics["condition_id"].astype(str) == str(condition))
            & (condition_metrics["summary_source"] == source)
        ]
        if subset.empty:
            continue
        row = subset.iloc[0]
        lines.append(
            f"{label}: r={_format_metric_value(row.get('r', np.nan))}, "
            f"R2={_format_metric_value(row.get('r2_corr', np.nan))}, "
            f"RMSE={_format_metric_value(row.get('rmse_z', np.nan))}"
        )
    indiv = simulation_metrics.loc[
        (simulation_metrics["condition_id"].astype(str) == str(condition))
        & (~simulation_metrics["is_posterior_mode"].astype(bool))
    ]
    if not indiv.empty:
        parts = []
        for metric, label in (("r", "r"), ("r2_corr", "R2"), ("rmse_z", "RMSE")):
            vals = pd.to_numeric(indiv[metric], errors="coerce").to_numpy(dtype=float)
            vals = vals[np.isfinite(vals)]
            if vals.size:
                parts.append(f"{label}={np.mean(vals):.2f}+/-{np.std(vals, ddof=0):.2f}")
            else:
                parts.append(f"{label}=nan")
        lines.append("Indiv sims: " + ", ".join(parts))
    return "\n".join(lines)


def _plot_condition_metric_violins(ax, condition, condition_metrics, simulation_metrics):
    metric_specs = [("r", "Pearson's r"), ("r2_corr", "R²")]
    condition_rows = simulation_metrics.loc[
        (simulation_metrics["condition_id"].astype(str) == str(condition))
        & (~simulation_metrics["is_posterior_mode"].astype(bool))
    ]
    datasets = []
    for metric, _ in metric_specs:
        values = pd.to_numeric(condition_rows[metric], errors="coerce").to_numpy(dtype=float)
        datasets.append(values[np.isfinite(values)])
    positions = np.arange(1, len(metric_specs) + 1, dtype=float)
    valid_positions = [pos for pos, values in zip(positions, datasets) if values.size]
    valid_datasets = [values for values in datasets if values.size]
    if valid_datasets:
        violin = ax.violinplot(valid_datasets, positions=valid_positions, widths=0.62, showmeans=False, showextrema=False, showmedians=False)
        valid_idx = 0
        for metric_idx, (metric, _) in enumerate(metric_specs):
            if datasets[metric_idx].size == 0:
                continue
            body = violin["bodies"][valid_idx]
            body.set_facecolor(FEATURE_BY_CONDITION_METRIC_COLORS[metric])
            body.set_edgecolor(FEATURE_BY_CONDITION_METRIC_COLORS[metric])
            body.set_alpha(0.20)
            body.set_linewidth(1.0)
            valid_idx += 1
    rng = np.random.default_rng(abs(hash(str(condition))) % (2**32))
    for pos, (metric, _), values in zip(positions, metric_specs, datasets):
        color = FEATURE_BY_CONDITION_METRIC_COLORS[metric]
        if values.size:
            jitter = rng.uniform(-0.08, 0.08, size=values.size)
            ax.scatter(
                np.full(values.shape, pos, dtype=float) + jitter,
                values,
                s=FEATURE_BY_CONDITION_METRIC_POINT_SIZE,
                alpha=FEATURE_BY_CONDITION_METRIC_POINT_ALPHA,
                color=color,
                edgecolors="none",
                zorder=3,
            )
            mean_value = float(np.mean(values))
            ax.hlines(
                mean_value,
                pos - 0.24,
                pos + 0.24,
                color=color,
                lw=FEATURE_BY_CONDITION_METRIC_MEAN_LW,
                zorder=4,
            )
        summary_subset = condition_metrics.loc[
            (condition_metrics["condition_id"].astype(str) == str(condition))
            & (condition_metrics["summary_source"] == "mean_feature")
        ]
        if not summary_subset.empty:
            summary_value = pd.to_numeric(pd.Series([summary_subset.iloc[0].get(metric)]), errors="coerce").iloc[0]
            if np.isfinite(summary_value):
                ax.scatter(
                    [pos],
                    [float(summary_value)],
                    marker=FEATURE_BY_CONDITION_METRIC_SUMMARY_MARKER,
                    s=FEATURE_BY_CONDITION_METRIC_SUMMARY_SIZE,
                    facecolors=color,
                    edgecolors="black",
                    linewidths=1.0,
                    zorder=5,
                )
    ax.set_xticks(positions)
    ax.set_xticklabels([label for _, label in metric_specs])
    # for tick, (metric, _) in zip(ax.get_xticklabels(), metric_specs):
    #     tick.set_color(FEATURE_BY_CONDITION_METRIC_COLORS[metric]) # commented out to keep black text
    ax.set_ylim(*map(float, FEATURE_BY_CONDITION_METRIC_VIOLIN_YLIM))
    ax.axhline(0.0, color="black", lw=1.0, ls="--", alpha=0.9, zorder=1)
    ax.axhline(1.0, color="0.55", lw=0.9, ls="--", alpha=0.85, zorder=1)
    ax.grid(False)
    ax.tick_params(axis="both", labelsize=11)
    ax.set_title("Feature-vector match", fontsize=10)
    metric_text = _condition_metric_text(condition, condition_metrics, simulation_metrics)
    if metric_text:
        ax.text(
            FEATURE_BY_CONDITION_METRIC_TEXT_X,
            FEATURE_BY_CONDITION_METRIC_TEXT_Y,
            metric_text,
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=FEATURE_BY_CONDITION_METRIC_TEXT_FONTSIZE,
            bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "edgecolor": "0.75", "alpha": 0.86},
            zorder=10,
        )
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)


def _plot_feature_values_by_condition(
    sim_table,
    exp_table,
    mean_feature_table,
    mean_trace_table,
    features,
    *,
    value_space,
    condition_metrics,
    simulation_metrics,
):
    features = [feature for feature in features if feature in sim_table.columns and feature in exp_table.columns]
    if not features:
        return None
    base_width = max(
        float(FEATURE_BY_CONDITION_MIN_FIG_WIDTH),
        float(FEATURE_BY_CONDITION_FIG_WIDTH_PER_FEATURE) * len(features),
    )
    fig_width = base_width + (3.0 if FEATURE_BY_CONDITION_SHOW_METRIC_VIOLIN_PANEL else 0.0)
    fig_height = float(FEATURE_BY_CONDITION_ROW_HEIGHT) * len(CONDITION_ORDER)
    if FEATURE_BY_CONDITION_SHOW_METRIC_VIOLIN_PANEL:
        fig, axes = plt.subplots(
            len(CONDITION_ORDER),
            2,
            figsize=(fig_width, fig_height),
            squeeze=False,
            gridspec_kw={
                "width_ratios": [FEATURE_BY_CONDITION_FEATURE_PANEL_WIDTH_RATIO, FEATURE_BY_CONDITION_METRIC_PANEL_WIDTH_RATIO],
                "wspace": 0.08,
            },
        )
    else:
        fig, axes_left = plt.subplots(len(CONDITION_ORDER), 1, figsize=(fig_width, fig_height), sharex=True, squeeze=False)
        axes = np.empty((len(CONDITION_ORDER), 2), dtype=object)
        axes[:, 0] = axes_left[:, 0]
        axes[:, 1] = None
    rng = np.random.default_rng(123)
    x_positions = np.arange(len(features), dtype=float)
    sampled = sim_table.loc[~sim_table["is_posterior_mode"].astype(bool)].copy()
    y_limits = None if FEATURE_BY_CONDITION_YLIM is None else tuple(map(float, FEATURE_BY_CONDITION_YLIM))
    for row_idx, condition in enumerate(CONDITION_ORDER):
        ax = axes[row_idx, 0]
        condition_color = FEATURE_MATCH_CONDITION_COLORS.get(condition, "#333333")
        condition_samples = sampled.loc[sampled["condition_id"].astype(str) == str(condition)]
        exp_subset = exp_table.loc[exp_table["condition_id"].astype(str) == str(condition)]
        mean_feature_subset = mean_feature_table.loc[mean_feature_table["condition_id"].astype(str) == str(condition)]
        mean_trace_subset = mean_trace_table.loc[mean_trace_table["condition_id"].astype(str) == str(condition)]
        for x, feature in zip(x_positions, features):
            feature_color = _feature_color(feature)
            mean_value_for_label = np.nan
            exp_value_for_label = np.nan
            y = pd.to_numeric(condition_samples.get(feature, pd.Series(dtype=float)), errors="coerce").to_numpy(dtype=float)
            y = y[np.isfinite(y)]
            if y.size:
                jitter = rng.uniform(-FEATURE_BY_CONDITION_POINT_JITTER, FEATURE_BY_CONDITION_POINT_JITTER, size=y.size)
                ax.scatter(
                    np.full(y.shape, x, dtype=float) + jitter,
                    y,
                    s=FEATURE_BY_CONDITION_POINT_SIZE,
                    alpha=FEATURE_BY_CONDITION_POINT_ALPHA,
                    color=feature_color,
                    edgecolors="none",
                    zorder=2,
                )
            if not mean_feature_subset.empty and feature in mean_feature_subset.columns:
                mean_value = pd.to_numeric(pd.Series([mean_feature_subset.iloc[0].get(feature)]), errors="coerce").iloc[0]
                mean_value_for_label = mean_value
                if np.isfinite(mean_value):
                    ax.hlines(
                        float(mean_value),
                        x - FEATURE_BY_CONDITION_MEAN_BAR_HALF_WIDTH,
                        x + FEATURE_BY_CONDITION_MEAN_BAR_HALF_WIDTH,
                        color=feature_color,
                        lw=FEATURE_BY_CONDITION_MEAN_BAR_LW,
                        alpha=0.98,
                        zorder=4,
                    )
            if FEATURE_BY_CONDITION_SHOW_MEAN_TRACE_BAR and not mean_trace_subset.empty and feature in mean_trace_subset.columns:
                mean_trace_value = pd.to_numeric(pd.Series([mean_trace_subset.iloc[0].get(feature)]), errors="coerce").iloc[0]
                if np.isfinite(mean_trace_value):
                    ax.hlines(
                        float(mean_trace_value),
                        x - FEATURE_BY_CONDITION_MEAN_BAR_HALF_WIDTH,
                        x + FEATURE_BY_CONDITION_MEAN_BAR_HALF_WIDTH,
                        color=feature_color,
                        lw=FEATURE_BY_CONDITION_MEAN_TRACE_BAR_LW,
                        ls="--",
                        alpha=0.98,
                        zorder=5,
                    )
            if not exp_subset.empty and feature in exp_subset.columns:
                exp_value = pd.to_numeric(pd.Series([exp_subset.iloc[0].get(feature)]), errors="coerce").iloc[0]
                exp_value_for_label = exp_value
                if np.isfinite(exp_value):
                    ax.hlines(
                        float(exp_value),
                        x - FEATURE_BY_CONDITION_EXPERIMENTAL_TARGET_HALF_WIDTH,
                        x + FEATURE_BY_CONDITION_EXPERIMENTAL_TARGET_HALF_WIDTH,
                        color=FEATURE_BY_CONDITION_EXPERIMENTAL_TARGET_COLOR,
                        lw=FEATURE_BY_CONDITION_EXPERIMENTAL_TARGET_LW,
                        alpha=0.98,
                        zorder=6,
                    )
            if np.isfinite(exp_value_for_label):
                label_lines = []
                label_lines.append(f"Abs. error:")
                if y.size:
                    abs_errors = np.abs(y - float(exp_value_for_label))
                    label_lines.append(f"indiv. {np.mean(abs_errors):.2f}+/-{np.std(abs_errors, ddof=0):.2f}")
                if np.isfinite(mean_value_for_label):
                    label_lines.append(f"mean {abs(float(mean_value_for_label) - float(exp_value_for_label)):.2f}")
                if label_lines:
                    ax.text(
                        x,
                        FEATURE_BY_CONDITION_FEATURE_ERROR_LABEL_Y,
                        "\n".join(label_lines),
                        transform=ax.get_xaxis_transform(),
                        ha="center",
                        va="top",
                        fontsize=FEATURE_BY_CONDITION_FEATURE_ERROR_LABEL_FONTSIZE,
                        color=feature_color,
                        bbox={"boxstyle": "round,pad=0.16", "facecolor": "white", "edgecolor": "none", "alpha": 0.72},
                        clip_on=False,
                        zorder=9,
                    )
        ax.set_ylabel(condition, color=condition_color, fontsize=12, fontweight="bold")
        ax.tick_params(axis="y", labelsize=9)
        ax.grid(False)
        ax.axhline(0.0, color="black", lw=1.0, ls="--", alpha=0.95, zorder=1)
        ax.axhline(-1.0, color="0.55", lw=0.9, ls="--", alpha=0.85, zorder=1)
        ax.axhline(1.0, color="0.55", lw=0.9, ls="--", alpha=0.85, zorder=1)
        if y_limits is not None:
            ax.set_ylim(*y_limits)
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)
        metric_ax = axes[row_idx, 1]
        if metric_ax is not None:
            _plot_condition_metric_violins(metric_ax, condition, condition_metrics, simulation_metrics)
            if row_idx < len(CONDITION_ORDER) - 1:
                metric_ax.set_xlabel("")
            if row_idx == 0:
                metric_ax.set_ylabel("Metric value")
            else:
                metric_ax.set_ylabel("")
    for row_idx in range(len(CONDITION_ORDER) - 1):
        axes[row_idx, 0].tick_params(axis="x", labelbottom=False)
    axes[-1, 0].set_xticks(x_positions)
    axes[-1, 0].set_xticklabels([_feature_tick_label(feature) for feature in features], rotation=70, ha="right", fontsize=7)
    for tick, feature in zip(axes[-1, 0].get_xticklabels(), features):
        tick.set_color(_feature_color(feature))
    from matplotlib.lines import Line2D
    feature_legend_handles = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor="#777777", markersize=6, alpha=FEATURE_BY_CONDITION_POINT_ALPHA, label="Posterior-predictive parameter sets"),
        Line2D([0], [0], color="#777777", lw=FEATURE_BY_CONDITION_MEAN_BAR_LW, label="Mean across individual simulations"),
        Line2D([0], [0], color=FEATURE_BY_CONDITION_EXPERIMENTAL_TARGET_COLOR, lw=FEATURE_BY_CONDITION_EXPERIMENTAL_TARGET_LW, label="Experimental condition mean (target)"),
    ]
    if FEATURE_BY_CONDITION_SHOW_MEAN_TRACE_BAR:
        feature_legend_handles.insert(2, Line2D([0], [0], color="#777777", lw=FEATURE_BY_CONDITION_MEAN_TRACE_BAR_LW, ls="--", label="Feature from mean curve"))
    metric_legend_handles = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor=FEATURE_BY_CONDITION_METRIC_COLORS["r"], markersize=6, alpha=FEATURE_BY_CONDITION_METRIC_POINT_ALPHA, label="Individual simulation metrics"),
        Line2D([0], [0], color=FEATURE_BY_CONDITION_METRIC_COLORS["r"], lw=FEATURE_BY_CONDITION_METRIC_MEAN_LW, label="Metric violin mean"),
        Line2D([0], [0], marker=FEATURE_BY_CONDITION_METRIC_SUMMARY_MARKER, color="none", markerfacecolor="#777777", markeredgecolor="black", markersize=8, label="Metric from mean feature vector"),
    ]
    fig.legend(handles=feature_legend_handles, loc="upper left", ncol=len(feature_legend_handles), frameon=True, bbox_to_anchor=(0.01, 0.995), title="Feature values")
    fig.legend(handles=metric_legend_handles, loc="upper right", ncol=len(metric_legend_handles), frameon=True, bbox_to_anchor=(0.99, 0.995), title="Metric violins")
    fig.suptitle(f"Posterior-predictive feature values by condition ({value_space})", y=1.012)
    fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.925), h_pad=2.8)
    return fig


condition_mean_feature_table_raw = _build_condition_mean_feature_table(sim_feature_table, shared_features)
condition_mean_feature_table_z = _zscore_feature_table(condition_mean_feature_table_raw, valid_scale_features)
condition_mean_feature_table_abs_zero = _abs_zero_scale_feature_table(condition_mean_feature_table_raw, valid_abs_zero_features)

summary_curve_feature_config = _feature_config_from_summary_table(sim_summary_df)
mean_trace_cache_path = Path(SUMMARY_CURVE_TRACE_CACHE_PATH)
load_if_available = SUMMARY_CURVE_FEATURE_SOURCE in {"load_or_compute", "load_only"}
if SUMMARY_CURVE_FEATURE_SOURCE == "load_only" and not mean_trace_cache_path.exists():
    raise FileNotFoundError(f"Requested SUMMARY_CURVE_FEATURE_SOURCE='load_only', but cache is missing: {mean_trace_cache_path}")
mean_trace_cache = build_posterior_predictive_condition_mean_traces(
    CONDITION_BATCH_ROOTS,
    summary_subdir_name=SUMMARY_SUBDIR_NAME,
    condition_order=CONDITION_ORDER,
    selected_parameter_set_ids=SELECTED_PARAMETER_SET_IDS,
    parameter_set_id_range=PARAMETER_SET_ID_RANGE,
    output_path=mean_trace_cache_path,
    load_if_available=load_if_available,
    force_recompute=(SUMMARY_CURVE_FEATURE_SOURCE == "compute" or FORCE_RECOMPUTE_SUMMARY_CURVES),
)
condition_mean_trace_feature_table_raw, mean_trace_generated_features = _extract_mean_trace_features(
    mean_trace_cache,
    shared_features,
    summary_curve_feature_config,
)
mean_trace_supported_features = [feature for feature in shared_features if feature in mean_trace_generated_features]
mean_trace_skipped_features = [feature for feature in shared_features if feature not in mean_trace_generated_features]
if mean_trace_skipped_features:
    warnings.warn(
        "Mean-trace summary metrics skip unsupported selected features: "
        + ", ".join(mean_trace_skipped_features[:12])
        + (" ..." if len(mean_trace_skipped_features) > 12 else "")
    )
condition_mean_trace_feature_table_z = _zscore_feature_table(condition_mean_trace_feature_table_raw, valid_scale_features)
condition_mean_trace_feature_table_abs_zero = _abs_zero_scale_feature_table(condition_mean_trace_feature_table_raw, valid_abs_zero_features)

summary_metric_table_by_space = {
    "raw": (condition_mean_feature_table_raw, exp_condition_feature_table, condition_mean_trace_feature_table_raw),
    "z": (condition_mean_feature_table_z, exp_condition_feature_table_z, condition_mean_trace_feature_table_z),
    "experimental_abs_zero": (condition_mean_feature_table_abs_zero, exp_condition_feature_table_abs_zero, condition_mean_trace_feature_table_abs_zero),
}
summary_metric_feature_table, summary_metric_exp_table, trace_metric_feature_table = summary_metric_table_by_space[metric_value_space]
summary_tables_for_condition_metrics = {
    "mean_feature": summary_metric_feature_table,
    "mean_trace": trace_metric_feature_table,
}
summary_level_condition_metrics = _build_summary_level_condition_metrics(
    summary_tables_for_condition_metrics,
    summary_metric_exp_table,
    metric_features,
)
summary_level_feature_metrics = _build_summary_level_feature_metrics(
    summary_tables_for_condition_metrics,
    summary_metric_exp_table,
    metric_features,
)
summary_level_feature_metrics.loc[
    (summary_level_feature_metrics["summary_source"] == "mean_trace")
    & (~summary_level_feature_metrics["feature"].isin(mean_trace_supported_features)),
    ["r", "r2_corr", "rmse_z", "mae_z", "n_conditions"],
] = np.nan

saved_feature_match_paths["condition_mean_feature_raw"] = _export_table(condition_mean_feature_table_raw, "posterior_predictive_condition_mean_feature_table_raw")
saved_feature_match_paths["condition_mean_feature_z"] = _export_table(condition_mean_feature_table_z, "posterior_predictive_condition_mean_feature_table_z")
saved_feature_match_paths["condition_mean_feature_abs_zero"] = _export_table(condition_mean_feature_table_abs_zero, "posterior_predictive_condition_mean_feature_table_experimental_abs_zero")
saved_feature_match_paths["condition_mean_trace_feature_raw"] = _export_table(condition_mean_trace_feature_table_raw, "posterior_predictive_condition_mean_trace_feature_table_raw")
saved_feature_match_paths["condition_mean_trace_feature_z"] = _export_table(condition_mean_trace_feature_table_z, "posterior_predictive_condition_mean_trace_feature_table_z")
saved_feature_match_paths["condition_mean_trace_feature_abs_zero"] = _export_table(condition_mean_trace_feature_table_abs_zero, "posterior_predictive_condition_mean_trace_feature_table_experimental_abs_zero")
saved_feature_match_paths["summary_level_condition_metrics"] = _export_table(summary_level_condition_metrics, "feature_match_summary_level_condition_metrics")

SUMMARY_SOURCE_ORDER = ["mean_feature", "mean_trace"]
SUMMARY_SOURCE_LABELS = {
    "mean_feature": "Mean of simulated features",
    "mean_trace": "Features from mean curves",
}
SUMMARY_SOURCE_COLORS = {
    "mean_feature": "#333333",
    "mean_trace": "#008C8C",
}
SUMMARY_SOURCE_MARKERS = {
    "mean_feature": "o",
    "mean_trace": "s",
}


def _plot_summary_level_condition_metrics(metrics_df):
    metric_specs = [
        ("r2_corr", "R2 corr"),
        ("r", "r"),
        ("rmse_z", "RMSE (z)"),
    ]
    fig, axes = plt.subplots(1, len(metric_specs), figsize=(15.0, 4.6), squeeze=False)
    x = np.arange(len(CONDITION_ORDER), dtype=float)
    for ax, (metric, title) in zip(axes.ravel(), metric_specs):
        for source in SUMMARY_SOURCE_ORDER:
            subset = (
                metrics_df.loc[metrics_df["summary_source"] == source]
                .set_index("condition_id")
                .reindex(CONDITION_ORDER)
            )
            y = pd.to_numeric(subset[metric], errors="coerce").to_numpy(dtype=float)
            ax.plot(
                x,
                y,
                marker=SUMMARY_SOURCE_MARKERS[source],
                color=SUMMARY_SOURCE_COLORS[source],
                lw=1.8,
                ms=7.0,
                label=SUMMARY_SOURCE_LABELS[source],
                zorder=3,
            )
        ax.set_title(title)
        ax.set_xticks(x)
        ax.set_xticklabels(CONDITION_ORDER, rotation=25, ha="right")
        for tick in ax.get_xticklabels():
            tick.set_color(FEATURE_MATCH_CONDITION_COLORS.get(tick.get_text(), "#333333"))
        ax.grid(axis="y", alpha=0.25)
        ax.set_xlabel("")
        ax.set_ylabel(title)
        if metric in {"r", "r2_corr"}:
            ax.axhline(0.0, color="0.55", lw=0.8, ls="--", alpha=0.7)
    axes.ravel()[0].legend(loc="best", frameon=True)
    fig.suptitle("Summary-level posterior-predictive match by condition")
    fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.94))
    return fig


def _plot_summary_level_feature_metrics(metrics_df):
    metric_specs = [
        ("r2_corr", "R2 corr"),
        ("r", "r"),
        ("rmse_z", "RMSE (z)"),
    ]
    feature_group_order = sorted(metrics_df["feature_group"].dropna().unique().tolist())
    fig, axes = plt.subplots(1, len(metric_specs), figsize=(18.0, 5.2), squeeze=False)
    for ax, (metric, title) in zip(axes.ravel(), metric_specs):
        finite_df = metrics_df.loc[np.isfinite(pd.to_numeric(metrics_df[metric], errors="coerce"))].copy()
        if finite_df.empty:
            ax.text(0.5, 0.5, "No finite values", ha="center", va="center", transform=ax.transAxes)
        else:
            sns.stripplot(
                data=finite_df,
                x="feature_group",
                y=metric,
                hue="summary_source",
                order=feature_group_order,
                hue_order=SUMMARY_SOURCE_ORDER,
                palette=SUMMARY_SOURCE_COLORS,
                dodge=True,
                jitter=0.22,
                alpha=0.75,
                size=5.0,
                ax=ax,
            )
        ax.set_title(title)
        ax.set_xlabel("")
        ax.set_ylabel(title)
        ax.tick_params(axis="x", rotation=30)
        ax.grid(axis="y", alpha=0.25)
        if metric in {"r", "r2_corr"}:
            ax.axhline(0.0, color="0.55", lw=0.8, ls="--", alpha=0.7)
        if ax.get_legend() is not None:
            ax.get_legend().remove()
    handles, labels = axes.ravel()[0].get_legend_handles_labels()
    if handles:
        label_lookup = {source: SUMMARY_SOURCE_LABELS.get(source, source) for source in SUMMARY_SOURCE_ORDER}
        labels = [label_lookup.get(label, label) for label in labels]
        fig.legend(handles, labels, loc="upper center", ncol=len(handles), frameon=True, bbox_to_anchor=(0.5, 0.98))
    fig.suptitle("Summary-level posterior-predictive match by feature")
    fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.90))
    return fig


# Older standalone metric/scatter figures are intentionally not generated; the combined figure below is the primary output.
summary_level_condition_fig_path = None
summary_level_feature_fig_path = None
scatter_fig_path = None

feature_by_condition_value_space = str(FEATURE_BY_CONDITION_VALUE_SPACE).lower()
if feature_by_condition_value_space not in {"raw", "z", "experimental_abs_zero"}:
    raise ValueError("FEATURE_BY_CONDITION_VALUE_SPACE must be 'raw', 'z', or 'experimental_abs_zero'")
feature_by_condition_table_by_space = {
    "raw": (sim_feature_table, exp_condition_feature_table, condition_mean_feature_table_raw, condition_mean_trace_feature_table_raw, shared_features),
    "z": (sim_feature_table_z, exp_condition_feature_table_z, condition_mean_feature_table_z, condition_mean_trace_feature_table_z, valid_scale_features),
    "experimental_abs_zero": (sim_feature_table_abs_zero, exp_condition_feature_table_abs_zero, condition_mean_feature_table_abs_zero, condition_mean_trace_feature_table_abs_zero, valid_abs_zero_features),
}
(
    feature_by_condition_sim_table,
    feature_by_condition_exp_table,
    feature_by_condition_mean_table,
    feature_by_condition_mean_trace_table,
    feature_by_condition_candidate_features,
) = feature_by_condition_table_by_space[feature_by_condition_value_space]
feature_by_condition_features = [
    feature for feature in feature_by_condition_candidate_features
    if feature in feature_by_condition_sim_table.columns and feature in feature_by_condition_exp_table.columns
]
feature_by_condition_fig = _plot_feature_values_by_condition(
    feature_by_condition_sim_table,
    feature_by_condition_exp_table,
    feature_by_condition_mean_table,
    feature_by_condition_mean_trace_table,
    feature_by_condition_features,
    value_space=feature_by_condition_value_space,
    condition_metrics=summary_level_condition_metrics,
    simulation_metrics=simulation_direction_metrics,
)
if feature_by_condition_fig is not None:
    feature_by_condition_fig_path = FEATURE_MATCH_OUTPUT_DIR / f"feature_match_values_by_condition_{feature_by_condition_value_space}.png"
    feature_by_condition_fig.savefig(feature_by_condition_fig_path, bbox_inches="tight")
    plt.close(feature_by_condition_fig)
else:
    feature_by_condition_fig_path = None

feature_match_summary_info.update({
    "summary_curve_trace_cache_path": str(mean_trace_cache_path),
    "summary_curve_feature_config": summary_curve_feature_config,
    "mean_trace_supported_features": mean_trace_supported_features,
    "mean_trace_skipped_features": mean_trace_skipped_features,
    "summary_level_condition_metrics_shape": tuple(summary_level_condition_metrics.shape),
    "summary_level_feature_metrics_shape": tuple(summary_level_feature_metrics.shape),
    "summary_level_condition_figure": None if summary_level_condition_fig_path is None else str(summary_level_condition_fig_path),
    "summary_level_feature_figure": None if summary_level_feature_fig_path is None else str(summary_level_feature_fig_path),
    "feature_value_scatter_figure": None if scatter_fig_path is None else str(scatter_fig_path),
    "feature_value_scatter_value_space": None,
    "feature_values_by_condition_value_space": feature_by_condition_value_space,
    "feature_values_by_condition_figure": None if feature_by_condition_fig_path is None else str(feature_by_condition_fig_path),
    "saved_tables": saved_feature_match_paths,
})
feature_match_summary_info, simulation_direction_metrics.head(), summary_level_condition_metrics.head()
